# 3 - Frequency study + multi-defect logo (frequency continuation)

Multiscale FWI: invert the 'L i square' logo at 50 -> 100 -> 200 kHz, each stage seeded by the previous (continuation avoids high-frequency cycle-skipping). The recovered model sharpens with frequency - the thesis result. Each inversion prints a classic training loop.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seidlr/acoustic-fwi-ndt-pytorch/blob/main/notebooks/03_frequency_study_and_logo.ipynb)

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    OWNER = "seidlr"  # change to your account if you forked
    !git clone -q https://github.com/{OWNER}/acoustic-fwi-ndt-pytorch.git
    %cd acoustic-fwi-ndt-pytorch
    sys.path.insert(0, os.path.abspath("src"))  # make `import fwi` importable now

import torch
from IPython.display import Image, display
from fwi.config import resolve_device, resolve_dtype

device = resolve_device()
dtype = resolve_dtype(device)
print("device:", device, "| dtype:", dtype)

## Frequency continuation 50 -> 100 -> 200 kHz

In [ ]:
from fwi.config import SimConfig
from fwi.problems import build_problem
from fwi.inversion import invert
from fwi import plotting

FREQS = [50_000., 100_000., 200_000.]
def corr(a,b):
    a=a.flatten().cpu().double(); b=b.flatten().cpu().double()
    a=a-a.mean(); b=b-b.mean()
    return float((a@b)/(a.norm()*b.norm()+1e-30))
ref = build_problem('logo', device=device, dtype=dtype, cfg=SimConfig(f0=FREQS[-1]))
true_update = ref.true_alpha2 - ref.start_alpha2
alpha2 = ref.start_alpha2.clone(); updates=[]
for f0 in FREQS:
    cfg = SimConfig(f0=f0)
    prob = build_problem('logo', device=device, dtype=dtype, cfg=cfg)
    print(f'=== {f0/1e3:.0f} kHz ===')
    alpha2, hist = invert(alpha2, prob.observed, prob.src_sig, prob.src_i, prob.src_j,
        prob.rec_i, prob.rec_j, cfg, active_mask=prob.active_mask, optimizer='lbfgs',
        n_iter=12, verbose=True)
    upd = (alpha2-ref.start_alpha2).detach(); updates.append(upd)
    print(f'   corr(true) = {corr(upd, true_update):.3f}')

## Inverted wave-speed update vs source frequency (sharpens with f0)

In [ ]:
display(Image(str(plotting.save_frequency_panel(updates, FREQS, 'outputs/nb3_freq.png'))))
display(Image(str(plotting.save_field(true_update, 'outputs/nb3_true.png', title='true logo'))))
display(Image(str(plotting.save_field(updates[-1], 'outputs/nb3_final.png',
    title='recovered logo (200 kHz)'))))

## More recordings -> better results: a single source moving from sensor to sensor

Fire one boundary sensor and reconstruct the logo, then fire each of several sensors in turn (round-robin / full-matrix capture) while the OTHER sensors record, summing the misfit and gradient over the shots. Both use the SAME moving-source acquisition and the SAME 50->100->200 kHz continuation - only the number of combined source positions differs. More positions illuminate the medium from more angles, so the multi-shot reconstruction correlates more strongly with the true logo - the thesis acquisition.

In [ ]:
from fwi.problems import build_multishot_problem
from fwi.inversion import invert_multishot

def run_multishot(n_shots, label):
    a = ref.start_alpha2.clone()
    for f0 in FREQS:
        cfg = SimConfig(f0=f0)
        mp = build_multishot_problem('logo', device=device, dtype=dtype, cfg=cfg, n_shots=n_shots)
        print(f'=== {label}: {f0/1e3:.0f} kHz ({len(mp.shots)} source position(s)) ===')
        a, _ = invert_multishot(a, mp.shots, mp.src_sig, cfg, active_mask=mp.active_mask,
            n_iter=10, verbose=True)
    return (a - ref.start_alpha2).detach(), len(mp.shots)

one_update, _ = run_multishot(1, '1 position')
many_update, n_pos = run_multishot(8, 'multi-shot')
corr_one = corr(one_update, true_update); corr_many = corr(many_update, true_update)
print(f'corr(true): 1 position {corr_one:.3f}  ->  {n_pos} positions {corr_many:.3f} '
      f'({corr_many - corr_one:+.3f})')
display(Image(str(plotting.save_field(one_update, 'outputs/nb3_single.png',
    title=f'1 source position (corr={corr_one:.2f})'))))
display(Image(str(plotting.save_field(many_update, 'outputs/nb3_multi.png',
    title=f'{n_pos} source positions (corr={corr_many:.2f})'))))

## Device speed benchmark (CPU vs GPU)

Run this notebook on a **CPU** runtime, then switch to a **GPU** runtime (Runtime -> Change runtime type -> T4 GPU) and run again. Compare the printed timings. A 104x204 sequential finite-difference solve launches many tiny kernels, so the GPU advantage may be modest - or even negative - at this grid size; that is itself the lesson. `torch.cuda.synchronize()` makes the GPU timing honest (CUDA kernels are asynchronous).

In [ ]:
import time
from fwi.config import SimConfig
from fwi.problems import build_problem
from fwi.forward import forward
from fwi.inversion import invert

def _sync():
    if device.type == 'cuda':
        torch.cuda.synchronize()

bench = build_problem('crack', device=device, dtype=dtype, cfg=SimConfig(f0=200_000.0))
def fwd():
    return forward(bench.start_alpha2, bench.src_sig, bench.src_i, bench.src_j,
                   bench.rec_i, bench.rec_j, bench.cfg)
fwd(); _sync()  # warmup: CUDA lazy init + autotune, not representative
K = 5
t0 = time.perf_counter()
for _ in range(K):
    fwd()
_sync(); fwd_ms = (time.perf_counter() - t0) / K * 1e3
t0 = time.perf_counter()
invert(bench.start_alpha2, bench.observed, bench.src_sig, bench.src_i, bench.src_j,
       bench.rec_i, bench.rec_j, bench.cfg, active_mask=bench.active_mask,
       optimizer='lbfgs', n_iter=10)
_sync(); inv_s = time.perf_counter() - t0
print(f'device = {device} | dtype = {dtype}')
print(f'forward solve : {fwd_ms:7.1f} ms/run (avg of {K})')
print(f'10-iter L-BFGS: {inv_s:7.2f} s')